In [1]:
# Cell 1 — Setup
import pandas as pd
import numpy as np
import re
from pathlib import Path
import os

# Adjust this if your notebook lives in notebooks/ (see Step 1)
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_EXTERNAL = PROJECT_ROOT / "data" / "external"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED dir:", DATA_PROCESSED)

PROJECT_ROOT: C:\Users\Anushka\OneDrive\Desktop\Project_Resume
PROCESSED dir: C:\Users\Anushka\OneDrive\Desktop\Project_Resume\data\processed


In [2]:
import os
from pathlib import Path
print("CWD:", os.getcwd())

CWD: C:\Users\Anushka\OneDrive\Desktop\Project_Resume\notebooks


In [3]:
import os
print("CWD:", os.getcwd())

CWD: C:\Users\Anushka\OneDrive\Desktop\Project_Resume\notebooks


In [4]:
# Cell 2 — Load raw resume data
resume_raw = pd.read_csv(DATA_RAW / "Resume.csv")
print("Raw shape:", resume_raw.shape)
print(resume_raw[["ID", "Category"]].head(3))

Raw shape: (2484, 4)
         ID Category
0  16852973       HR
1  22323967       HR
2  33176873       HR


In [16]:
# Cell 3 — Resume text cleaning (conservative)

# Cell 3 — Resume text cleaning (conservative) + drop empties

def clean_resume_text(text: str) -> str:
    """
    Conservative cleaning for resume text.
    - Removes invisible / zero-width unicode characters
    - Collapses runs of whitespace into single spaces
    - Preserves technical tokens: C++, C#, .NET, Node.js, React.js, AWS, SQL
    - Does NOT remove punctuation, stopwords, or lowercase (that's Phase 6)
    """
    if not isinstance(text, str):
        return ""
    # Strip invisible / zero-width characters (defensive — PDFs can emit these)
    text = re.sub(r"[\u200b\u200c\u200d\u200e\u200f\ufeff\u202a-\u202e]", "", text)
    text = re.sub(r"[\r\t]+", " ", text)
    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


resume_clean = resume_raw.copy()
resume_clean["Resume_str"] = resume_clean["Resume_str"].apply(clean_resume_text)

# Drop rows that collapsed to empty (whitespace-only resumes)
before = len(resume_clean)
resume_clean = resume_clean[resume_clean["Resume_str"].str.len() > 0].reset_index(drop=True)
after = len(resume_clean)
print(f"Dropped {before - after} whitespace-only row(s).")
print(f"New shape: {resume_clean.shape}")

# Sanity checks
print("Empty resumes after drop:", (resume_clean["Resume_str"].str.len() == 0).sum())
print("Min length:", resume_clean["Resume_str"].str.len().min())
print("Max length:", resume_clean["Resume_str"].str.len().max())

Dropped 1 whitespace-only row(s).
New shape: (2483, 5)
Empty resumes after drop: 0
Min length: 688
Max length: 36643


In [6]:
# Cell 4 — Save cleaned resume dataset
out_path = DATA_PROCESSED / "resumes_clean.csv"
resume_clean.to_csv(out_path, index=False)
print("Saved to:", out_path)
print("File size:", round(out_path.stat().st_size / 1024, 1), "KB")

Saved to: C:\Users\Anushka\OneDrive\Desktop\Project_Resume\data\processed\resumes_clean.csv
File size: 54076.7 KB


In [7]:
# Cell 5 — Load and trim LinkedIn postings
postings_path = DATA_RAW / "linkedin_jobs" / "postings.csv"

# Load only the columns we'll use — big speedup
USECOLS = [
    "job_id", "company_name", "title", "description",
    "location", "formatted_experience_level", "skills_desc",
]

# Read first N rows (streaming nrows to stay memory-friendly)
postings = pd.read_csv(postings_path, usecols=USECOLS, nrows=20000)
print("Loaded for inspection:", postings.shape)

# Drop rows with no description (unusable for matching)
postings = postings.dropna(subset=["description"]).reset_index(drop=True)
print("After dropping null descriptions:", postings.shape)

# Sample down to 5000 for our working dataset (deterministic seed for reproducibility)
postings_sample = postings.sample(n=min(5000, len(postings)), random_state=42).reset_index(drop=True)
print("Final sample shape:", postings_sample.shape)
print("\nMissing values:\n", postings_sample.isnull().sum())

Loaded for inspection: (20000, 7)
After dropping null descriptions: (20000, 7)
Final sample shape: (5000, 7)

Missing values:
 job_id                           0
company_name                   114
title                            0
description                      0
location                         0
formatted_experience_level    1476
skills_desc                   4945
dtype: int64


In [8]:
# Cell 6 — Clean job description text
def clean_jd_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[\r\t]+", " ", text)
    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

postings_sample["description_clean"] = postings_sample["description"].apply(clean_jd_text)

# Stats
print("Description length describe (chars):")
print(postings_sample["description_clean"].str.len().describe())

# Show sample
print("\nSample cleaned JD (first 300 chars):")
print(postings_sample["description_clean"].iloc[0][:300])

Description length describe (chars):
count     5000.000000
mean      3745.623600
std       2194.346853
min         23.000000
25%       2132.750000
50%       3382.000000
75%       4966.750000
max      21027.000000
Name: description_clean, dtype: float64

Sample cleaned JD (first 300 chars):
ExperienceUndergraduate degree in computer science, engineering, mathematics or equivalent education/experience and 8+ years of experience in IT.5+ years of hands-on experience of building solutions using Microsoft Dynamics 365, Power Platform product. Preferred QualificationsExperience in the Whole


In [9]:
# Cell 7 — Save trimmed postings
out_path = DATA_PROCESSED / "postings_sample.csv"
postings_sample.to_csv(out_path, index=False)
print("Saved to:", out_path)
print("File size:", round(out_path.stat().st_size / 1024 / 1024, 2), "MB")

Saved to: C:\Users\Anushka\OneDrive\Desktop\Project_Resume\data\processed\postings_sample.csv
File size: 36.29 MB


In [10]:
# Cell 8 — ESCO skills → normalization table
esco_raw = pd.read_csv(DATA_EXTERNAL / "skills_en.csv")
print("Raw ESCO shape:", esco_raw.shape)

# Keep only useful columns
ESCO_COLS = ["conceptUri", "preferredLabel", "altLabels", "skillType", "description", "reuseLevel"]
esco = esco_raw[ESCO_COLS].copy()

# Drop rows missing preferred label (should be 0)
esco = esco.dropna(subset=["preferredLabel"]).reset_index(drop=True)

# Fill altLabels with empty string
esco["altLabels"] = esco["altLabels"].fillna("")

print("Cleaned ESCO shape:", esco.shape)
print("Sample rows:")
print(esco.head(3))

Raw ESCO shape: (13960, 13)
Cleaned ESCO shape: (13960, 6)
Sample rows:
                                          conceptUri  \
0  http://data.europa.eu/esco/skill/0005c151-5b5a...   
1  http://data.europa.eu/esco/skill/00064735-8fad...   
2  http://data.europa.eu/esco/skill/000709ed-2be5...   

                      preferredLabel  \
0               manage musical staff   
1  supervise correctional procedures   
2    apply anti-oppressive practices   

                                           altLabels         skillType  \
0  manage music staff\ncoordinate duties of music...  skill/competence   
1  manage prison procedures\nmonitor correctional...  skill/competence   
2  make use of anti-oppressive practices\nuse ant...  skill/competence   

                                         description           reuseLevel  
0  Assign and manage staff tasks in areas such as...      sector-specific  
1  Supervise the operations of a correctional fac...  occupation-specific  
2  Identify oppre

In [11]:
# Cell 9 — Build variant → canonical mapping

def build_alias_map(esco_df: pd.DataFrame) -> dict:
    """
    Returns a dict: {lowercased_variant: preferredLabel}
    Includes the preferred label itself, plus all alt labels.
    """
    alias_map = {}
    for _, row in esco_df.iterrows():
        canonical = str(row["preferredLabel"]).strip()
        if not canonical:
            continue
        alias_map[canonical.lower()] = canonical
        alts = row["altLabels"]
        if isinstance(alts, str) and alts:
            for alt in alts.split("\n"):
                alt = alt.strip()
                if alt:
                    alias_map.setdefault(alt.lower(), canonical)
    return alias_map

alias_map = build_alias_map(esco)
print("Total aliases:", len(alias_map))
print("Sample entries:")
for k in list(alias_map)[:10]:
    print(f"  {k!r} -> {alias_map[k]!r}")

Total aliases: 99624
Sample entries:
  'manage musical staff' -> 'manage musical staff'
  'manage music staff' -> 'manage musical staff'
  'coordinate duties of musical staff' -> 'manage musical staff'
  'direct musical staff' -> 'manage musical staff'
  'manage staff of music' -> 'manage musical staff'
  'supervise correctional procedures' -> 'supervise correctional procedures'
  'manage prison procedures' -> 'supervise correctional procedures'
  'monitor correctional procedures' -> 'supervise correctional procedures'
  'oversee prison procedures' -> 'supervise correctional procedures'
  'oversee correctional procedures' -> 'supervise correctional procedures'


In [12]:
# Cell 10 — Save ESCO artifacts
import json

esco.to_csv(DATA_PROCESSED / "esco_skills_clean.csv", index=False)
with open(DATA_PROCESSED / "esco_alias_map.json", "w", encoding="utf-8") as f:
    json.dump(alias_map, f, ensure_ascii=False, indent=2)

print("Saved:", DATA_PROCESSED / "esco_skills_clean.csv")
print("Saved:", DATA_PROCESSED / "esco_alias_map.json")

Saved: C:\Users\Anushka\OneDrive\Desktop\Project_Resume\data\processed\esco_skills_clean.csv
Saved: C:\Users\Anushka\OneDrive\Desktop\Project_Resume\data\processed\esco_alias_map.json


In [13]:
# Cell 11 — Reload and verify
r = pd.read_csv(DATA_PROCESSED / "resumes_clean.csv")
p = pd.read_csv(DATA_PROCESSED / "postings_sample.csv")
e = pd.read_csv(DATA_PROCESSED / "esco_skills_clean.csv")
with open(DATA_PROCESSED / "esco_alias_map.json", encoding="utf-8") as f:
    am = json.load(f)

print("resumes_clean:", r.shape)
print("postings_sample:", p.shape)
print("esco_skills_clean:", e.shape)
print("alias_map size:", len(am))

resumes_clean: (2484, 4)
postings_sample: (5000, 8)
esco_skills_clean: (13960, 6)
alias_map size: 99624


In [14]:
# Inspect the empty row
resume_raw["len_before"] = resume_raw["Resume_str"].astype(str).str.len()
resume_clean["len_after"] = resume_clean["Resume_str"].astype(str).str.len()

# Which row(s) had length 0 (or nearly 0)?
print("Raw rows with length <= 5:", (resume_raw["len_before"] <= 5).sum())
print("Clean rows with length == 0:", (resume_clean["len_after"] == 0).sum())

# Show the offending row(s)
bad = resume_clean[resume_clean["len_after"] == 0]
print("\nBad row(s):")
print(bad[["ID", "Category"]])
print("\nCategory distribution of the offending row:", bad["Category"].tolist())

Raw rows with length <= 5: 0
Clean rows with length == 0: 1

Bad row(s):
           ID              Category
656  12632728  BUSINESS-DEVELOPMENT

Category distribution of the offending row: ['BUSINESS-DEVELOPMENT']


In [15]:
# Find the raw row
raw_row = resume_raw[resume_raw["ID"] == 12632728]
print("Raw row count:", len(raw_row))

raw_text = raw_row["Resume_str"].iloc[0]

# Three key observations
print("Type:", type(raw_text))
print("Raw length:", len(raw_text))
print("Repr (first 200 chars):", repr(raw_text[:200]))
print("Unique chars (first 50):", sorted(set(raw_text))[:50])
print("Is all whitespace?:", raw_text.isspace() if isinstance(raw_text, str) else "not str")
print("Is NaN?:", raw_text != raw_text)  # NaN != NaN is True

Raw row count: 1
Type: <class 'str'>
Raw length: 21
Repr (first 200 chars): '                     '
Unique chars (first 50): [' ']
Is all whitespace?: True
Is NaN?: False
